In [1]:
import gradio as gr
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage import util

# --- Helper: Histogram Plot ---
def plot_histogram(img):
    if img is None: return None
    plt.figure(figsize=(5, 3))
    plt.hist(img.flatten(), bins=256, color='gray', alpha=0.7)
    plt.title("Intensity Distribution", fontsize=8)
    plt.xticks(fontsize=6); plt.yticks(fontsize=6)
    plt.tight_layout(); plt.savefig("hist_temp.png"); plt.close()
    return "hist_temp.png"

# --- TAB 1: Restoration & Enhancement ---
def dynamic_restoration(src, n_type, n_lvl, p_method, gamma, f_type, k_size, d0):
    if src is None: return [None]*4 + ["Waiting for Image..."]
    src = cv2.resize(src, (350, 350))
    gray_orig = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY) if len(src.shape) == 3 else src

    # 1. Optional Degradation (Noise Injection)
    noisy = gray_orig.copy()
    if n_type == "Salt & Pepper":
        # TODO: Implement Salt & Pepper noise logic here
        sp = util.random_noise(gray_orig.astype(np.float32) / 255, mode="s&p",amount=n_lvl).astype(np.float32)
        noisy = (sp * 255).astype(np.uint8)
    elif n_type == "Gaussian":
        # TODO: Implement Gaussian noise logic here
        sigma = n_lvl
        noise = np.random.normal(0, sigma, gray_orig.shape).astype(np.float32)
        noisy = np.clip(gray_orig.astype(np.float32) / 255 + noise, 0, 1)
        noisy = (noisy * 255).astype(np.uint8)
    elif n_type == "Periodic":
        rows, cols = gray_orig.shape
        x = np.arange(cols)
        noise = n_lvl * 255 * np.sin(2 * np.pi * x / 20)
        noise2d = np.tile(noise, (rows, 1)).astype(np.float32)
        noisy = np.clip(gray_orig.astype(np.float32) + noise2d,0 ,255 ).astype(np.uint8)
    elif n_type == "Uniform":
        noise = np.random.uniform(-n_lvl * 255, n_lvl * 255, gray_orig.shape).astype(np.float32)
        noisy = np.clip(gray_orig.astype(np.float32) + noise ,0 , 255).astype(np.uint8)
    elif n_type == "Exponential":
        noise = np.random.exponential(scale=n_lvl * 255, size=gray_orig.shape).astype(np.float32)
        noisy = np.clip(gray_orig.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # 2. Intensity Transformations (Enhancement)
    work = noisy.astype(np.float32)
    if p_method == "Negative":
        work = 255 - work # Example implemented
    elif p_method == "Log":
        c=255/(np.log(1+np.max(work)))
        work = c * np.log(1+work)
    elif p_method == "Gamma":
        work=255*((work/255.0)**gamma)
    elif p_method == "Histogram Equalization":
         work=cv2.equalizeHist(noisy.astype(np.uint8))

    processed = np.clip(work, 0, 255).astype(np.uint8)

    # 3. Filtering (Restoration)(Arwa)
    restored = processed.copy()
    if f_type != "None":
        k = int(k_size) | 1
        #Gaussian Noise\Exponential Noise
        if f_type == "Mean":
            restored = cv2.blur(processed, (k, k)) # Example implemented
        #Salt and Paper Noise
        elif f_type == "Median":
            restored = cv2.medianBlur(processed, k) # Example implemented
        #Uniform Noise
        elif f_type == "Midpoint":
            kernel = np.ones((k, k), np.uint8)
            minf=cv2.erode(processed,kernel)
            maxf=cv2.dilate(processed,kernel)
            restored=0.5*(minf.astype(np.float32)+maxf.astype(np.float32))
            restored=np.clip(restored,0,255).astype(np.uint8)
         # Laplacian Sharpening
        elif f_type == "Laplacian":
            laplacian = cv2.Laplacian(processed, cv2.CV_64F)
            restored = cv2.convertScaleAbs(processed - laplacian)#(Spatial Domain End Here (Arwa))
        #  frequency domain filtering Starts here (Rand Part)
        elif f_type in ["Low Pass", "High Pass", "Band Pass", "Notch"]:
              rows, cols = processed.shape
              # STEP 1: FFT & shift to center
              f = np.fft.fft2(processed.astype(np.float32))
              fshift = np.fft.fftshift(f)

              # STEP 2: Build distance map from center
              crow, ccol = rows // 2, cols // 2
              u = np.arange(rows) - crow
              v = np.arange(cols) - ccol
              V, U = np.meshgrid(v, u)
              D = np.sqrt(U**2 + V**2)  # Euclidean distance from center

              # STEP 3: Build filter mask
              if f_type == "Low Pass":
                 # Ideal LPF: pass all frequencies within D0
                 H = (D <= d0).astype(np.float32)

              elif f_type == "High Pass":
                 # Ideal HPF: block all frequencies within D0
                 H = (D >= d0).astype(np.float32)

              elif f_type == "Band Pass":
                 # Passes a band between D0 and a second cutoff (D0*1.8)
                 d1 = d0 * 1.8
                 H = ((D >= d0) & (D <= d1)).astype(np.float32)


              elif f_type == "Notch":
                 width = max(d0 * 0.25, 5)
                 H = np.ones((rows, cols), np.float32)
                 H[np.abs(D - d0) < width] = 0

              # STEP 4: Apply mask in frequency domain
              fshift_filtered = fshift * H

              # STEP 5: Visualize spectrum (log scale) for debugging
              magnitude_spectrum = np.log(1 + np.abs(fshift))
              magnitude_filtered = np.log(1 + np.abs(fshift_filtered))

              # STEP 6: Inverse FFT → back to spatial domain
              f_ishift = np.fft.ifftshift(fshift_filtered)
              img_back = np.fft.ifft2(f_ishift)
              restored = np.abs(img_back)
              restored = np.clip(restored, 0, 255).astype(np.uint8) #frequency domain filtering Ends here (Rand Part)


    # 4. Quantitative Metrics
    mse_val = np.mean((gray_orig.astype(np.float32) - restored.astype(np.float32))**2)
    psnr = 100 if mse_val == 0 else 20 * np.log10(255.0 / np.sqrt(mse_val))

    return restored, cv2.absdiff(gray_orig, restored), plot_histogram(restored), f"MSE: {mse_val:.2f} | PSNR: {psnr:.2f} dB"

# --- TAB 2: Segmentation & Morphology ---
#LAMEES
def dynamic_segmentation(img, seg_meth, threshold, morph_op, se_shape, se_size, class_label):
    if img is None: return [None]*3
    img = cv2.resize(img, (350, 350))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img

    # 1. Segmentation
    if seg_meth == "Global":
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY) # Example implemented
        # TODO: Implement Adaptive or Otsu Thresholding here
    elif seg_meth == "Adaptive":
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2) #Adaptive
    else:
        binary = np.zeros_like(gray)


    # 2. Morphology
    s_size = int(se_size)
    # TODO: Implement Structuring Element (SE) logic for Square, Cross, and Disk
    if se_shape == "Triangle":
        se =np.array([
            [0, 1, 0],
            [1, 1, 1],
            [1, 1, 1]
        ], dtype=np.uint8)

    elif se_shape == "Circle":
        se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (s_size, s_size))

    elif se_shape == "Disk":
        se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (s_size, s_size))
    else:
        se = np.ones((s_size, s_size), np.uint8)

    m_out = binary.copy()
    if morph_op == "Erosion":
        m_out = cv2.erode(binary, se) # Example implemented
    elif morph_op in ["Dilation", "Opening", "Closing", "Boundary Extraction"]:
        # TODO: Implement the selected Morphological operation here
        if morph_op == "Dilation":
            m_out = cv2.dilate(binary, se)

        elif morph_op == "Opening":
            m_out = cv2.morphologyEx(binary, cv2.MORPH_OPEN, se)

        elif morph_op == "Closing":
            m_out = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, se)

        elif morph_op == "Boundary Extraction":
          erosion = cv2.erode(binary, se)
          m_out = cv2.bitwise_xor(binary, erosion)


    # 3. Feature Extraction (Rana)
    contours, _ = cv2.findContours(m_out,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) > 0:
        contours = [max(contours, key=cv2.contourArea)]
    feats = []
    for i, c in enumerate(contours):
        area = cv2.contourArea(c)
        if area < 50: continue
        perim = cv2.arcLength(c, True)
        # TODO: Implement Circularity/Compactness formula here
        circ =  (4 * np.pi * area) / (perim ** 2) if perim != 0 else 0 # Calculate circularity feature
        compactness = (perim ** 2) / area if area != 0 else 0 # Calculate compactness feature
        num_labels, labels_im = cv2.connectedComponents(m_out) # Compute Euler number using connected components
        euler = num_labels - 1
        feats.append({"Area": round(area,1), "Perimeter": round(perim,1), "Circularity": round(circ,3),"Compactness":round(compactness,3),"Euler":euler,"Class": class_label})

    df = pd.DataFrame(feats)
    df.to_csv("current_sample.csv", index=False)
    return m_out, df, "current_sample.csv"

# --- TAB 3: Batch Analysis (Logic remains for evaluation) ---(Rana)
def run_classification_analysis(train_file, test_file):
    try:
        df_tr = pd.read_csv(train_file.name) if train_file.name.endswith('.csv') else pd.read_excel(train_file.name)
        df_ts = pd.read_csv(test_file.name) if test_file.name.endswith('.csv') else pd.read_excel(test_file.name)
        cols = ['Area', 'Perimeter', 'Circularity','Compactness','Euler']
        scaler = StandardScaler()
        X_all = scaler.fit_transform(np.vstack((df_tr[cols].values, df_ts[cols].values)))
        pca = PCA(n_components=2); X_pca = pca.fit_transform(X_all) # preform PCA

        plt.figure(figsize=(5, 4))  # Plot PCA feature distribution
        if 'Class' in df_tr.columns:
            for cls in df_tr['Class'].unique():
                idx = df_tr['Class'] == cls
                plt.scatter(
                    X_pca[:len(df_tr), 0][idx],
                    X_pca[:len(df_tr), 1][idx],
                    label=f"Train Class {cls}",alpha=0.6)

        else: # Visualize training classes in PCA space
           plt.scatter(
               X_pca[:len(df_tr), 0],
               X_pca[:len(df_tr), 1],
               label='Train Refs',
               alpha=0.6 )
        plt.scatter( # Plot test sample position
            X_pca[len(df_tr):, 0],
            X_pca[len(df_tr):, 1],
            c='red',
            marker='x')
        plt.title("PCA Cluster Visualization"); plt.legend(); plt.grid(True, alpha=0.2); plt.tight_layout()
        plt.savefig("pca_plot.png"); plt.close()

        preds = []
        for vec in df_ts[cols].values:
            dists = [np.linalg.norm(vec - tr_vec) for tr_vec in df_tr[cols].values]
            preds.append(df_tr.iloc[np.argmin(dists)]['Class'] if 'Class' in df_tr.columns else "Label Missing")
        df_ts['Prediction'] = preds
        return df_ts, "pca_plot.png", f"Success: {len(df_ts)} samples predicted."
    except Exception as e: return None, None, f"Error: {str(e)}"

# --- UI Layout  ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DIAPF: Dynamic Image Analysis & Processing Framework")
    with gr.Tabs():
        with gr.TabItem("1. Restoration & Enhancement"):
            with gr.Row():
                with gr.Column(scale=2):
                    src_img = gr.Image(label="Source Image", type="numpy", height=350, width=350)
                    mse_out = gr.Textbox(label="Metrics (MSE & PSNR)")
                with gr.Column(scale=3, variant="panel"):
                    gr.Markdown("### Phase 1 Control")
                    with gr.Accordion("Noise Injection", open=False):
                        n_t = gr.Dropdown(["None", "Salt & Pepper", "Gaussian", "Periodic", "Uniform", "Exponential"], label="Noise Type", value="None"); n_l = gr.Slider(0, 0.2, 0.05, label="Intensity")
                    with gr.Accordion("Filters & Enhancement", open=True):
                        p_m = gr.Radio(["None", "Negative", "Log", "Gamma", "Histogram Equalization"], label="Transformation", value="None"); gam = gr.Slider(0.1, 5.0, 1.0, label="Gamma")
                        f_t = gr.Dropdown(["None", "Mean", "Median", "Midpoint","Laplacian","Notch","Low Pass", "High Pass","Band Pass"], label="Filter", value="None"); k_s = gr.Slider(1, 15, 3, step=2, label="Kernel Size"); d_0 = gr.Slider(1, 200, 50, label="D0 Cutoff")
                with gr.Column(scale=2):
                    res_out = gr.Image(label="Restored", height=350, width=350); dif_out = gr.Image(label="Diff Map", height=200, width=350); his_out = gr.Image(label="Histogram", height=150, width=350)
            t1_in = [src_img, n_t, n_l, p_m, gam, f_t, k_s, d_0]; [i.change(dynamic_restoration, t1_in, [res_out, dif_out, his_out, mse_out]) for i in t1_in]

        with gr.TabItem("2. Segmentation & Morphology"):
            with gr.Row():
                with gr.Column(scale=1):
                    seg_src = gr.Image(label="Input Image", height=350, width=350)
                    gr.Markdown("### Phase 2: Structural Extraction")
                    sm = gr.Radio(["Global", "Adaptive"], label="Method", value="Global"); st = gr.Slider(0, 255, 127, label="Threshold")
                    mo = gr.Dropdown(["None", "Erosion", "Dilation", "Opening", "Closing", "Boundary Extraction"], label="Operation", value="None")
                    sh = gr.Radio(["Triangle", "Circle", "Disk"], label="Shape", value="Triangle"); sz = gr.Slider(3, 15, 3, step=2, label="Size")
                with gr.Column(scale=1):
                    bin_out = gr.Image(label="Binary Mask", height=350, width=350); feat_out = gr.DataFrame(label="Features Vector", interactive=True)
                    gr.Markdown("> **IMPORTANT NOTE:** Use the 'Class Name' field below *only* for Training References. Leave it blank for Test Samples.")
                    c_lbl = gr.Textbox(label="Class Name (Required for Training Only)", placeholder="e.g. Coin_A", interactive=True)
                    file_out = gr.File(label="Download CSV Record")
            t2_in = [seg_src, sm, st, mo, sh, sz, c_lbl]; [i.change(dynamic_segmentation, t2_in, [bin_out, feat_out, file_out]) for i in t2_in]

        with gr.TabItem("3. Classification & PCA"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Phase 3: Analytical Analysis")
                    ftr = gr.File(label="Upload Training Reference (CSV)"); fts = gr.File(label="Upload Final Test Samples (CSV)")
                    bc = gr.Button("Run Analysis", variant="primary")
                with gr.Column():
                    pp = gr.Image(label="PCA Mapping"); cr = gr.Textbox(label="Status Report")
            ct = gr.DataFrame(label="Final Results Table"); bc.click(run_classification_analysis, [ftr, fts], [ct, pp, cr])

demo.launch()

/tmp/ipykernel_26136/2305487276.py:256: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6719b086c7d08f1f05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# New Section